In [1]:
import polars as pl

# Data

Initial Date is 15 september 2019 (75 days)
Centroid

In [2]:
df = pl.read_csv("../../data/yjmob/yjmob100k-dataset1.csv.gz")
df

uid,d,t,x,y
i64,i64,i64,i64,i64
0,0,1,79,86
0,0,2,79,86
0,0,8,77,86
0,0,9,77,86
0,0,19,81,89
…,…,…,…,…
99999,74,38,119,77
99999,74,39,132,94
99999,74,40,124,105


In [9]:
df = df.filter(pl.col("uid") < 500).to_pandas()
df

,uid,d,t,x,y
0,0,0,1,79,86
1,0,0,2,79,86
2,0,0,8,77,86
3,0,0,9,77,86
4,0,0,19,81,89
...,...,...,...,...,...
613225,499,74,29,176,36
613226,499,74,32,175,38
613227,499,74,33,176,35
613228,499,74,36,177,38


In [10]:
from datetime import datetime, timedelta
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Polygon

# Assuming 'df' is your loaded 111M+ row DataFrame containing columns: [uid, d, t, x, y]


def recover_yjmob_geometries(df: pd.DataFrame) -> gpd.GeoDataFrame:
    print("Step 1: Reconstructing absolute calendar timeline...")
    # Base chronological anchor: Sunday, September 15, 2019
    base_date = datetime(2019, 9, 15)

    # Convert d (days) and t (30-minute intervals) into exact timestamps
    df["timestamp"] = base_date + pd.to_timedelta(
        df["d"], unit="D"
    ) + pd.to_timedelta(df["t"] * 30, unit="m")

    print("Step 2: Undoing geometric matrix transformations...")
    # Invert the R90(Fy(D)) transformation highlighted in Section 4.2
    # This restores proper cardinal orientation axes
    x_oriented = df["y"]
    y_oriented = -df["x"]

    # Calculate standard mean offsets to isolate grid-relative steps
    x_mean = x_oriented.mean()
    y_mean = y_oriented.mean()

    print("Step 3: Correcting aspect-ratio stretch and step scaling...")
    # Grid steps are exactly 500 meters
    STEP_SIZE = 500.0

    # Cancel out the code's 1.1x and 0.9x structural distortions
    x_metric_deltas = ((x_oriented - x_mean) / 1.1) * STEP_SIZE
    y_metric_deltas = ((y_oriented - y_mean) / 0.9) * STEP_SIZE

    print("Step 4: Mapping to Japan Plane Rectangular CS VII (EPSG:2449)...")
    # Metric coordinate values corresponding to the paper's fine-tuned WGS84 anchor
    ANCHOR_X_2449 = -11610.0
    ANCHOR_Y_2449 = -104950.0

    # Project coordinates out to exact metric bounds
    df["metric_x"] = ANCHOR_X_2449 + x_metric_deltas
    df["metric_y"] = ANCHOR_Y_2449 + y_metric_deltas

    print("Step 5: Generating bounding cell polygons...")
    half_step = STEP_SIZE / 2.0

    # Construct the 500m x 500m square cell bounds for every point directly in metric space
    polygons = [
        Polygon(
            [
                (mx - half_step, my - half_step),
                (mx + half_step, my - half_step),
                (mx + half_step, my + half_step),
                (mx - half_step, my + half_step),
                (mx - half_step, my - half_step),
            ]
        )
        for mx, my in zip(df["metric_x"], df["metric_y"])
    ]

    # Initialize the GeoDataFrame directly in Japan's metric coordinate projection
    gdf_metric = gpd.GeoDataFrame(
        df[["uid", "timestamp", "d", "t"]], geometry=polygons, crs=2449
    )

    print("Step 6: Reprojecting to standard WGS84 (Lat/Lon)...")
    # Safely transform the metric plane projections back to standard global GPS coordinates
    gdf_wgs84 = gdf_metric.to_crs(epsg=4326)

    # Extract distinct Latitude/Longitude centroid positions for simple pointwise mapping
    centroids = gdf_wgs84.geometry.centroid
    gdf_wgs84["lon"] = centroids.x
    gdf_wgs84["lat"] = centroids.y

    # Reorder columns logically
    gdf_wgs84 = gdf_wgs84[
        ["uid", "timestamp", "lat", "lon", "geometry", "d", "t"]
    ]

    print("Processing complete.")
    return gdf_wgs84


# Run the recovery pipeline
recovered_gdf = recover_yjmob_geometries(df)
recovered_gdf

Step 1: Reconstructing absolute calendar timeline...
Step 2: Undoing geometric matrix transformations...
Step 3: Correcting aspect-ratio stretch and step scaling...
Step 4: Mapping to Japan Plane Rectangular CS VII (EPSG:2449)...
Step 5: Generating bounding cell polygons...
Step 6: Reprojecting to standard WGS84 (Lat/Lon)...


/tmp/ipykernel_2320174/428797164.py:74: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf_wgs84.geometry.centroid


Processing complete.


,uid,timestamp,lat,lon,geometry,d,t
0,0,2019-09-15 00:30:00,35.258568,137.024751,"POLYGON ((137.02201 35.25631, 137.0275 35.2563...",0,1
1,0,2019-09-15 01:00:00,35.258568,137.024751,"POLYGON ((137.02201 35.25631, 137.0275 35.2563...",0,2
2,0,2019-09-15 04:00:00,35.268584,137.024734,"POLYGON ((137.02199 35.26633, 137.02749 35.266...",0,8
3,0,2019-09-15 04:30:00,35.268584,137.024734,"POLYGON ((137.02199 35.26633, 137.02749 35.266...",0,9
4,0,2019-09-15 09:30:00,35.248569,137.039753,"POLYGON ((137.03701 35.24631, 137.0425 35.2463...",0,19
...,...,...,...,...,...,...,...
613225,499,2019-11-28 14:30:00,34.772236,136.777289,"POLYGON ((136.77457 34.76997, 136.78003 34.769...",74,29
613226,499,2019-11-28 16:00:00,34.777276,136.787198,"POLYGON ((136.78448 34.77501, 136.78994 34.775...",74,32
613227,499,2019-11-28 16:30:00,34.772220,136.772323,"POLYGON ((136.7696 34.76996, 136.77506 34.7699...",74,33
613228,499,2019-11-28 18:00:00,34.767259,136.787244,"POLYGON ((136.78452 34.765, 136.78999 34.76501...",74,36
